In [1]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score

In [2]:
train_df = pd.read_csv("../data/churn_train.csv")
test_df = pd.read_csv("../data/churn_test.csv")

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

train_df.head()

FileNotFoundError: [Errno 2] No such file or directory: '../data/churn_train.csv'

In [ ]:
train_df.info()
test_df.info()

In [ ]:
train_df = train_df.drop_duplicates()
test_df = test_df.drop_duplicates()

train_df.columns = train_df.columns.str.strip().str.replace(" ", "_")
test_df.columns = test_df.columns.str.strip().str.replace(" ", "_")

print(train_df.columns)

In [ ]:
train_df["Churn"] = train_df["Churn"].astype(str).str.strip().str.lower().map({"yes": 1, "no": 0})
test_df["Churn"] = test_df["Churn"].astype(str).str.strip().str.lower().map({"yes": 1, "no": 0})

print(train_df["Churn"].value_counts())
print(test_df["Churn"].value_counts())

In [ ]:
X_train = train_df.drop("Churn", axis=1).copy()
y_train = train_df["Churn"].copy()

X_test = test_df.drop("Churn", axis=1).copy()
y_test = test_df["Churn"].copy()

In [ ]:
label_encoders = {}

for col in X_train.columns:
    if X_train[col].dtype == "object":
        X_train[col] = X_train[col].fillna(X_train[col].mode()[0])
        
        # fill missing test values using train mode
        X_test[col] = X_test[col].fillna(X_train[col].mode()[0])

        le = LabelEncoder()
        le.fit(X_train[col].astype(str))

        X_train[col] = le.transform(X_train[col].astype(str))

        # handle unseen categories in test set
        test_values = X_test[col].astype(str)
        unseen_mask = ~test_values.isin(le.classes_)

        if unseen_mask.any():
            test_values = test_values.copy()
            test_values[unseen_mask] = le.classes_[0]

        X_test[col] = le.transform(test_values)

        label_encoders[col] = le

    else:
        median_val = X_train[col].median()
        X_train[col] = X_train[col].fillna(median_val)
        X_test[col] = X_test[col].fillna(median_val)

In [ ]:
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42
)

model.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nROC-AUC:", roc_auc_score(y_test, y_prob))

In [ ]:
Both CSVs must have the same columns

Run this:

In [ ]:
print("Train columns:", train_df.columns.tolist())
print("Test columns:", test_df.columns.tolist())